# LegacyScribe — Fine-tuning Notebook
**Qwen3.5-9B · Unsloth + LoRA · Modal A100 · W&B monitoring**

Run order:
1. Install dependencies
2. Load and inspect datasets
3. Stratified 80/20 split → `train.jsonl` + `val.jsonl`
4. Upload datasets to Modal volume
5. Define and run fine-tuning on Modal
6. Monitor with W&B
7. Export LoRA adapter + GGUF

---
## Cell 1 — Install dependencies

In [ ]:
# Run once. Restart kernel after if prompted.
!pip install modal wandb datasets --quiet
!modal setup   # opens browser login if first time

In [ ]:
# import os
# import wandb

# # Load from environment variable if set, otherwise fall back to interactive prompt.
# # Set it once in your terminal: export WANDB_API_KEY=your_key_here
# # Or add it to a .env file and load with python-dotenv.

# api_key = os.environ.get("WANDB_API_KEY")

# if api_key:
#     wandb.login(key=api_key, relogin=True)
#     print("W&B logged in from WANDB_API_KEY env var.")
# else:
#     print("WANDB_API_KEY not found in environment — falling back to interactive login.")
#     wandb.login()   # prompts for key in notebook

---
## Cell 2 — Config (edit these)

In [ ]:
import os

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = "./datasets"   # local folder containing the five subfolders

DATASET_FILES = {
    "questioner":  os.path.join(DATASET_ROOT, "questioner",  "combined_dataset.jsonl"),
    "arcdetector": os.path.join(DATASET_ROOT, "arcdetector", "combined_dataset.jsonl"),
    "extractor":   os.path.join(DATASET_ROOT, "extractor",   "combined_dataset.jsonl"),
    "reconciler":  os.path.join(DATASET_ROOT, "reconciler",  "combined_dataset.jsonl"),
    "publisher":   os.path.join(DATASET_ROOT, "publisher",   "combined_dataset.jsonl"),
}

TRAIN_OUT  = "./train.jsonl"
VAL_OUT    = "./val.jsonl"

# ── Split ────────────────────────────────────────────────────────────────────
VAL_RATIO  = 0.20
SEED       = 42

# ── Model ────────────────────────────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # swap to 9B if your Modal tier allows
LORA_RANK  = 16

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS          = 3
BATCH_SIZE      = 4
GRAD_ACCUM      = 4       # effective batch = 4 × 4 = 16
LR              = 2e-4
MAX_SEQ_LEN     = 2048
WARMUP_STEPS    = 20

# ── W&B ──────────────────────────────────────────────────────────────────────
WANDB_PROJECT   = "legacyscribe"
WANDB_RUN_NAME  = "qwen25-lora-r16-run1"

# ── Modal ────────────────────────────────────────────────────────────────────
MODAL_APP_NAME  = "legacyscribe-finetune"
MODAL_VOLUME    = "legacyscribe-data"
MODAL_GPU       = "A100-80GB"   # or "A10G" if budget is tight (slower, ~$4 less)
MODAL_TIMEOUT   = 60 * 60 * 4  # 4 hours max

print("Config loaded.")

---
## Cell 3 — Load datasets and inspect

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

def load_jsonl(path: str) -> list:
    """Load a .jsonl file, skipping blank lines. Returns list of dicts."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ⚠ JSON error in {path} line {i}: {e}")
    return rows

def validate_example(ex: dict, name: str, idx: int) -> bool:
    """Basic schema check: must have messages list with system/user/assistant."""
    msgs = ex.get("messages", [])
    if not isinstance(msgs, list) or len(msgs) < 2:
        print(f"  ⚠ {name}[{idx}]: missing or short messages list")
        return False
    roles = [m.get("role") for m in msgs]
    if "user" not in roles or "assistant" not in roles:
        print(f"  ⚠ {name}[{idx}]: missing user or assistant role")
        return False
    # Check no empty assistant response
    for m in msgs:
        if m.get("role") == "assistant" and not m.get("content", "").strip():
            print(f"  ⚠ {name}[{idx}]: empty assistant content")
            return False
    return True

# ── Load all datasets ─────────────────────────────────────────────────────────
raw = {}   # name → list of examples (with _source tag injected)
total = 0

for name, path in DATASET_FILES.items():
    if not Path(path).exists():
        print(f"❌  {name}: file not found at {path}")
        continue
    examples = load_jsonl(path)
    valid    = []
    for i, ex in enumerate(examples):
        if validate_example(ex, name, i):
            ex["_source"] = name   # tag for stratification (stripped before save)
            valid.append(ex)
    skipped = len(examples) - len(valid)
    raw[name] = valid
    total += len(valid)
    status = "✅" if not skipped else "⚠ "
    print(f"{status} {name:12s}: {len(valid):3d} valid  ({skipped} skipped)")

print(f"\nTotal clean examples: {total}")

---
## Cell 4 — Deduplication check

In [ ]:
def fingerprint(ex: dict) -> str:
    """Hash on user+assistant content only (ignore _source tag)."""
    msgs = ex.get("messages", [])
    key  = "".join(
        m.get("content", "").strip()
        for m in msgs
        if m.get("role") in ("user", "assistant")
    )
    return key

seen   = set()
dupes  = 0

for name, examples in raw.items():
    clean = []
    for ex in examples:
        fp = fingerprint(ex)
        if fp in seen:
            dupes += 1
        else:
            seen.add(fp)
            clean.append(ex)
    raw[name] = clean

total_after = sum(len(v) for v in raw.values())
print(f"Duplicates removed: {dupes}")
print(f"Clean total after dedup: {total_after}")

---
## Cell 5 — Stratified 80/20 split

In [ ]:
import math

random.seed(SEED)

train_examples = []
val_examples   = []

print(f"{'Dataset':<14} {'Total':>6} {'Train':>6} {'Val':>6}")
print("-" * 36)

for name, examples in raw.items():
    shuffled  = examples.copy()
    random.shuffle(shuffled)

    n_val     = math.ceil(len(shuffled) * VAL_RATIO)
    n_train   = len(shuffled) - n_val

    val_examples.extend(shuffled[:n_val])
    train_examples.extend(shuffled[n_val:])

    print(f"{name:<14} {len(shuffled):>6} {n_train:>6} {n_val:>6}")

# Final shuffle so datasets aren't grouped by source
random.shuffle(train_examples)
random.shuffle(val_examples)

print("-" * 36)
print(f"{'TOTAL':<14} {len(train_examples)+len(val_examples):>6} {len(train_examples):>6} {len(val_examples):>6}")

# Verify stratification
print("\nSource distribution in val set:")
val_sources = Counter(ex["_source"] for ex in val_examples)
for src, cnt in sorted(val_sources.items()):
    print(f"  {src:<14}: {cnt}")

---
## Cell 6 — Save train.jsonl and val.jsonl

In [ ]:
def save_jsonl(examples: list, path: str):
    """Save examples, stripping the internal _source tag."""
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            clean = {k: v for k, v in ex.items() if k != "_source"}
            f.write(json.dumps(clean, ensure_ascii=False) + "\n")
    print(f"Saved {len(examples)} examples → {path}")

save_jsonl(train_examples, TRAIN_OUT)
save_jsonl(val_examples,   VAL_OUT)

print(f"\nReady: {TRAIN_OUT}  ({len(train_examples)} rows)")
print(f"Ready: {VAL_OUT}  ({len(val_examples)} rows)")

---
## Cell 7 — Spot-check 5 random training examples

In [ ]:
sample = random.sample(train_examples, min(5, len(train_examples)))
for i, ex in enumerate(sample, 1):
    src   = ex.get("_source", "?")
    msgs  = ex["messages"]
    user  = next((m["content"] for m in msgs if m["role"] == "user"),      "")
    asst  = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
    print(f"── Example {i} ({src}) ──────────────────")
    print(f"USER : {user[:120]}...")
    print(f"ASST : {asst[:120]}...")
    print()

---
## Cell 8 — Create Modal volume and upload datasets

In [ ]:
import subprocess

# Create volume (safe to re-run — skips if exists)
subprocess.run(["modal", "volume", "create", MODAL_VOLUME], check=False)

# Upload both split files
for local_path in [TRAIN_OUT, VAL_OUT]:
    result = subprocess.run(
        ["modal", "volume", "put", MODAL_VOLUME, local_path, "/data/"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅  Uploaded {local_path} → Modal volume /{MODAL_VOLUME}/data/")
    else:
        print(f"❌  Upload failed: {result.stderr}")

---
## Cell 9 — W&B login

In [ ]:
import wandb
wandb.login(key=api_key, relogin=True)  # paste your API key from wandb.ai/authorize

---
## Cell 10 — Define Modal fine-tuning app

This cell writes the full training script and registers it as a Modal function.
Nothing runs yet — execution happens in Cell 11.

In [ ]:
import modal

app    = modal.App(MODAL_APP_NAME)
volume = modal.Volume.from_name(MODAL_VOLUME)

# ── Container image ───────────────────────────────────────────────────────────
# Unsloth publishes a pre-built image with CUDA, torch, and unsloth already in.
image = (
    modal.Image.from_registry(
        "unsloth/unsloth-studio:latest",
        add_python="3.11",
    )
    .pip_install(
        "wandb",
        "datasets",
        "trl>=0.8.6",
        "transformers>=4.40",
        "peft>=0.10",
    )
)

# ── Training function ─────────────────────────────────────────────────────────
@app.function(
    image=image,
    gpu=MODAL_GPU,
    timeout=MODAL_TIMEOUT,
    volumes={f"/data": volume},
    secrets=[
        modal.Secret.from_name("wandb-secret"),     # store WANDB_API_KEY here
        modal.Secret.from_name("huggingface-secret"), # store HF_TOKEN here
    ],
)
def train(
    base_model:    str = BASE_MODEL,
    lora_rank:     int = LORA_RANK,
    epochs:        int = EPOCHS,
    batch_size:    int = BATCH_SIZE,
    grad_accum:    int = GRAD_ACCUM,
    lr:          float = LR,
    max_seq_len:   int = MAX_SEQ_LEN,
    warmup_steps:  int = WARMUP_STEPS,
    wandb_project: str = WANDB_PROJECT,
    wandb_run:     str = WANDB_RUN_NAME,
):
    import os, json, wandb
    from unsloth import FastModel
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig

    # ── W&B init ──────────────────────────────────────────────────────────────
    wandb.init(project=wandb_project, name=wandb_run)

    # ── Load model + tokenizer ────────────────────────────────────────────────
    # FastModel (not FastLanguageModel) because Qwen2.5 registers as a vision
    # model and FastLanguageModel will throw a type error on load.
    model, tokenizer = FastModel.from_pretrained(
        model_name=base_model,
        max_seq_length=max_seq_len,
        dtype=None,         # auto-selects bf16 on A100
        load_in_4bit=False, # we fine-tune full bf16, then quantize to GGUF
    )

    # ── Attach LoRA adapter ───────────────────────────────────────────────────
    model = FastModel.get_peft_model(
        model,
        r=lora_rank,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=lora_rank * 2,   # alpha = 2 × rank is a reliable default
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
    )

    # ── Load datasets from volume ─────────────────────────────────────────────
    def load_jsonl(path):
        rows = []
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    train_rows = load_jsonl("/data/train.jsonl")
    val_rows   = load_jsonl("/data/val.jsonl")

    print(f"Train: {len(train_rows)} examples")
    print(f"Val:   {len(val_rows)} examples")

    # ── Format as chat strings ────────────────────────────────────────────────
    # Unsloth's SFTTrainer expects a 'text' column with the full formatted prompt.
    def format_example(row):
        return tokenizer.apply_chat_template(
            row["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

    train_dataset = Dataset.from_list(
        [{"text": format_example(r)} for r in train_rows]
    )
    val_dataset   = Dataset.from_list(
        [{"text": format_example(r)} for r in val_rows]
    )

    # ── Training config ───────────────────────────────────────────────────────
    # Total steps = ceil(train_size / (batch × accum)) × epochs
    steps_per_epoch = len(train_rows) // (batch_size * grad_accum)
    total_steps     = steps_per_epoch * epochs
    print(f"Steps per epoch: {steps_per_epoch}  |  Total steps: {total_steps}")

    config = SFTConfig(
        output_dir="/data/checkpoints",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        bf16=True,
        optim="adamw_8bit",          # memory-efficient
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=steps_per_epoch // 2,  # eval twice per epoch
        save_strategy="steps",
        save_steps=steps_per_epoch,
        save_total_limit=2,           # keep best + latest
        load_best_model_at_end=True,  # loads checkpoint with lowest val loss
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_seq_length=max_seq_len,
        dataset_text_field="text",
        packing=False,                # keep examples separate for val accuracy
        report_to="wandb",
        run_name=wandb_run,
        seed=42,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=config,
    )

    # ── Train ─────────────────────────────────────────────────────────────────
    print("\n Starting training...")
    trainer.train()

    # ── Save LoRA adapter ─────────────────────────────────────────────────────
    adapter_path = "/data/lora_adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"\n LoRA adapter saved → {adapter_path}")

    # ── Merge + export GGUF ───────────────────────────────────────────────────
    # Merge LoRA weights into the base model, then quantize to Q4_K_M.
    # The resulting .gguf file is what llama.cpp loads directly.
    print("\n Merging LoRA and exporting Q4_K_M GGUF...")
    model.save_pretrained_gguf(
        "/data/gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("\n GGUF saved → /data/gguf")

    wandb.finish()
    return {"status": "done", "adapter": adapter_path, "gguf": "/data/gguf"}


print("Modal app defined. Run Cell 11 to start training.")

---
## Cell 11 — Launch training on Modal

> **Cost estimate:** A100-80GB on Modal is ~\$3.70/hr.
> Expected runtime: 2–3 hours → **\$8–12** from your \$200 credit.
>
> Watch the W&B dashboard live at https://wandb.ai while this runs.
> If val loss plateaus before epoch 3, kill the cell — the best checkpoint is already saved.

In [ ]:
# This streams logs directly into the notebook output.
with modal.enable_output():
    with app.run():
        result = train.remote()

print("\nTraining complete.")
print(result)

---
## Cell 12 — Download GGUF from Modal volume

In [ ]:
import subprocess, os

os.makedirs("./output", exist_ok=True)

# Download the GGUF folder from the Modal volume
result = subprocess.run(
    ["modal", "volume", "get", MODAL_VOLUME, "/data/gguf", "./output/gguf"],
    capture_output=True, text=True
)

if result.returncode == 0:
    gguf_files = list(Path("./output/gguf").glob("*.gguf"))
    print(f"✅  Downloaded {len(gguf_files)} GGUF file(s):")
    for f in gguf_files:
        size_gb = f.stat().st_size / 1e9
        print(f"    {f.name}  ({size_gb:.1f} GB)")
else:
    print(f"❌  Download failed:\n{result.stderr}")

---
## Cell 13 — Smoke test (local, optional)

Requires `llama-cpp-python` installed locally. Runs a quick inference check
on each of the five agent profiles to confirm the model responds correctly
before you deploy to Gradio.

In [ ]:
# !pip install llama-cpp-python --quiet

from pathlib import Path
from llama_cpp import Llama

gguf_path = next(Path("./output/gguf").glob("*.gguf"), None)
if gguf_path is None:
    print("No GGUF found in ./output/gguf — run Cell 12 first.")
else:
    print(f"Loading {gguf_path.name}...")
    llm = Llama(
        model_path=str(gguf_path),
        n_ctx=2048,
        n_gpu_layers=-1,   # offload all layers to GPU/Metal
        verbose=False,
    )

    SMOKE_TESTS = [
        {
            "profile": "extractor",
            "system": "You are an extractor agent. Given a memory fragment, extract the person reference as JSON with keys: who, relationship, how_referenced. Output only valid JSON, nothing else.",
            "user":   "My father used to wake before the sun every morning to go to the fields.",
            "check":  lambda r: r.strip().startswith("{"),
        },
        {
            "profile": "questioner",
            "system": "You are a gentle memory guide helping an elderly person tell their life story. Ask exactly one warm, open follow-up question. Never ask more than one question.",
            "user":   "I remember the soup my mother made during the monsoon.",
            "check":  lambda r: r.count("?") == 1,
        },
        {
            "profile": "arcdetector",
            "system": "You are an arc detector agent. Given a memory fragment, identify the narrative stage. Output one word only: setup, tension, turn, or meaning.",
            "user":   "We had always had enough rice. That year was different — the field dried up before harvest.",
            "check":  lambda r: r.strip().lower() in ("setup", "tension", "turn", "meaning"),
        },
        {
            "profile": "reconciler",
            "system": "You are a reconciler agent. Given two memory notes, detect if they contradict each other. Output JSON with keys: conflict, conflict_type, note_a_claim, note_b_claim, resolution. Output only valid JSON, nothing else.",
            "user":   "Note A: My father came back from India during the monsoon.\nNote B: My father returned from India just after Dashain.",
            "check":  lambda r: r.strip().startswith("{"),
        },
        {
            "profile": "publisher",
            "system": "You are a publisher agent. Given 3–5 atomic memory notes, synthesize them into a single warm narrative paragraph suitable for a family memory book. Write in first person. Output only the paragraph, nothing else.",
            "user":   "Notes:\n1. My mother made sel roti every Tihar.\n2. The whole house smelled of mustard oil.\n3. My sister and I raced to light the first diya.",
            "check":  lambda r: len(r.strip()) > 100,
        },
    ]

    all_pass = True
    for test in SMOKE_TESTS:
        messages = [
            {"role": "system",    "content": test["system"]},
            {"role": "user",      "content": test["user"]},
        ]
        response = llm.create_chat_completion(
            messages=messages,
            max_tokens=256,
            temperature=0.3,
        )
        reply = response["choices"][0]["message"]["content"]
        passed = test["check"](reply)
        icon   = "✅" if passed else "❌"
        all_pass = all_pass and passed
        print(f"{icon}  [{test['profile']:12s}] {reply[:100].strip()}")

    print()
    print("All smoke tests passed ✅" if all_pass else "Some tests failed ❌ — review before deploying")

---
## Cell 14 — Evaluation metrics

Runs the automatic checks from Step 4 of the pipeline:
- Val loss already logged to W&B (target < 1.2)
- JSON parse rate for extractor + reconciler (target > 95%)
- Single-question rate for questioner (target > 90%)
- Arc output validity (target > 95%)

In [ ]:
import json

val_examples_raw = []
with open(VAL_OUT, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            val_examples_raw.append(json.loads(line))

# Group by agent type using system prompt keywords
def detect_agent(ex):
    sys_msg = next(
        (m["content"] for m in ex["messages"] if m["role"] == "system"), ""
    ).lower()
    if "extractor"   in sys_msg: return "extractor"
    if "reconciler"  in sys_msg: return "reconciler"
    if "arc detector" in sys_msg or "narrative stage" in sys_msg: return "arcdetector"
    if "question"    in sys_msg: return "questioner"
    if "publisher"   in sys_msg: return "publisher"
    return "unknown"

metrics = {
    "extractor":   {"json_ok": 0, "total": 0},
    "reconciler":  {"json_ok": 0, "total": 0},
    "arcdetector": {"valid_word": 0, "total": 0},
    "questioner":  {"one_q": 0, "total": 0},
    "publisher":   {"long_enough": 0, "total": 0},
}

ARC_VALID = {"setup", "tension", "turn", "meaning"}

for ex in val_examples_raw:
    agent  = detect_agent(ex)
    gold   = next(
        (m["content"] for m in ex["messages"] if m["role"] == "assistant"), ""
    ).strip()

    if agent == "extractor":
        metrics["extractor"]["total"] += 1
        try:
            json.loads(gold)
            metrics["extractor"]["json_ok"] += 1
        except Exception:
            pass

    elif agent == "reconciler":
        metrics["reconciler"]["total"] += 1
        try:
            json.loads(gold)
            metrics["reconciler"]["json_ok"] += 1
        except Exception:
            pass

    elif agent == "arcdetector":
        metrics["arcdetector"]["total"] += 1
        if gold.lower() in ARC_VALID:
            metrics["arcdetector"]["valid_word"] += 1

    elif agent == "questioner":
        metrics["questioner"]["total"] += 1
        if gold.count("?") == 1:
            metrics["questioner"]["one_q"] += 1

    elif agent == "publisher":
        metrics["publisher"]["total"] += 1
        if len(gold) > 100:
            metrics["publisher"]["long_enough"] += 1

# ── Print report ──────────────────────────────────────────────────────────────
print("Validation set — gold-label quality check (not model predictions)")
print("─" * 55)
for agent, m in metrics.items():
    total = m["total"]
    if total == 0:
        continue
    key  = [k for k in m if k != "total"][0]
    rate = m[key] / total * 100
    icon = "✅" if rate >= 95 else ("⚠ " if rate >= 80 else "❌")
    print(f"{icon}  {agent:<14}: {key} = {m[key]}/{total} ({rate:.0f}%)")

print()
print("NOTE: these check gold labels, not model outputs.")
print("Re-run on model predictions after training for real eval.")

---
## Cell 15 — Next steps

After all cells complete:

```
./output/gguf/*.gguf   ← drop this into your llama.cpp Gradio Space
./train.jsonl          ← keep for reproducibility
./val.jsonl            ← keep for reproducibility
```

**W&B dashboard** — check:
- `eval/loss` should be below 1.2 at epoch 3
- Loss curve should still be decreasing, not flat — if flat before epoch 3,
  you stopped at the right time
- No spike at epoch 2 (would indicate overfitting on 850 examples)

**If eval/loss stays above 1.4:**
1. Review the 20 worst-loss examples in W&B
2. Fix or remove them from the dataset
3. Re-run training — costs another ~$10

**GGUF file size reference:**
- Qwen2.5-7B Q4_K_M ≈ 4.4 GB  (fits in Gradio 5 GB Space limit)
- Qwen2.5-9B Q4_K_M ≈ 5.8 GB  (needs Gradio persistent storage upgrade)

**Hackathon demo tip:**
Preload the system prompt prefix into llama.cpp's `--cache-prompt` flag.
First inference will be slow; all subsequent ones will be fast.